# GLEE Competition agent V3 — adaptive percentile optimizer

This notebook is built from the live [GLEE `llms.txt`](https://glee-competition.com/llms.txt) specification (August 25,
2026) and the repository's V2 agent. It targets **payoff percentile**, not just
agreement rate: it captures surplus when the opponent is flexible, but prices in
the severe rating cost of no-deals and invalid moves.

V3 adds bounded opponent models, continuation-value decisions, hidden-value
concession inference, direct recommendation-precision learning, and a finite
credibility budget for persuasion. No strategy can guarantee a higher rating:
opponents, configurations, and rating updates are stochastic. Test in bounded
family-specific batches and keep the version whose live results are stronger.

Run cells in order. Never run two Play cells with the same API key.



In [ ]:
%pip install -q -U glee-sdk



## API key, imports, and thread-safe memory

Named opponents get cross-game profiles. Hidden opponents get game-local memory.
Stable hashing makes mixed strategies reproducible and concurrency-safe.



In [ ]:
import hashlib
import math
import os
import statistics
import threading
from collections import defaultdict, deque
from getpass import getpass

os.environ["GLEE_API_KEY"] = getpass("GLEE API key: ")

LOCK = threading.RLock()
SEEN = set()
DECISION_LOG = deque(maxlen=1000)

BARGAINING_MEMORY = defaultdict(lambda: {
    "rejected_floor": 0.0, "opponent_demands": deque(maxlen=30)
})
NEGOTIATION_MEMORY = defaultdict(lambda: {
    "seller_prices": deque(maxlen=40), "buyer_prices": deque(maxlen=40)
})
PERSUASION_MEMORY = defaultdict(lambda: {
    "pos_high": 0.0, "pos_low": 0.0,
    "neg_high": 0.0, "neg_low": 0.0,
    "positive_buys": 0.0, "positive_decisions": 0.0,
})



## Shared schema helpers



In [ ]:
def clamp(x, low, high):
    return max(low, min(high, x))

def finite_float(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else float(default)
    except (TypeError, ValueError):
        return float(default)

def round_progress(state):
    current = max(1, int(state.get("round", 1)))
    maximum = state.get("max_rounds")
    if state.get("horizon_known") and maximum:
        return clamp((current - 1) / max(1, int(maximum) - 1), 0.0, 1.0)
    # Unknown horizons should concede slowly: there is no announced deadline.
    return min(0.72, (current - 1) / 12.0)

def final_round(state):
    return bool(state.get("horizon_known") and state.get("max_rounds") and
                int(state.get("round", 1)) >= int(state["max_rounds"]))

def player_index(player):
    return 1 if player in {"player_1", "alice"} else 2

def canonical_player(player):
    return f"player_{player_index(player)}"

def other_player(player):
    return "player_2" if player_index(player) == 1 else "player_1"

def opponent_key(game, family):
    opponent = game.get("opponent") or {}
    if opponent.get("type") != "hidden" and opponent.get("name"):
        return f"{family}:named:{opponent.get('type')}:{opponent['name']}"
    return f"{family}:game:{game.get('game_id', '')}"

def stable_unit(game, salt=""):
    state = game.get("game_state") or {}
    token = f"{game.get('game_id', '')}:{state.get('round', 1)}:{salt}"
    value = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "big")
    return value / 2**64

def logistic(x):
    return 1.0 / (1.0 + math.exp(-clamp(x, -60.0, 60.0)))

def allocation(offer, player):
    keys = (("player_1_gain", "alice_gain") if player_index(player) == 1 else
            ("player_2_gain", "bob_gain"))
    for key in keys:
        if key in offer:
            return finite_float(offer[key], None)
    return None

def action_message(text):
    return str(text)[:2000]

def signal_polarity(value):
    if isinstance(value, dict):
        value = value.get("decision", value.get("message"))
    if value is None:
        return None
    text = str(value).strip().lower().replace("_", " ")
    negative_phrases = (
        "do not buy", "don't buy", "do not recommend", "don't recommend",
        "not worth", "pass", "avoid", "low quality", "bad product",
    )
    positive_phrases = (
        "buy", "recommend", "worth it", "high quality", "great product",
        "good product", "positive",
    )
    if text in {"no", "false", "negative", "not recommended"}:
        return False
    if any(phrase in text for phrase in negative_phrases):
        return False
    if text in {"yes", "true", "recommended"}:
        return True
    if any(phrase in text for phrase in positive_phrases):
        return True
    return None



## 1. Bargaining — demand learning plus continuation value

Rejections provide a lower bound on the responder's acceptable share. Opponent
proposals reveal their aspiration and concession path. V3 combines those signals
with the visible Rubinstein benchmark, then searches splits by probability-weighted
payoff. Acceptance compares the current share with a discounted, risk-adjusted next
proposal—not with a fixed percentage of the pot.



In [ ]:
def player_delta(state, player, default=0.93):
    return clamp(finite_float(state.get(f"delta_{player_index(player)}", default), default),
                 0.01, 0.999)

def rubinstein_responder_share(proposer_delta, responder_delta):
    denominator = 1.0 - proposer_delta * responder_delta
    proposer_share = ((1.0 - responder_delta) / denominator
                      if denominator > 1e-10 else 0.5)
    return clamp(1.0 - proposer_share, 0.05, 0.95)

def update_bargaining_memory(game, me, opponent, money):
    key = opponent_key(game, "bargaining")
    with LOCK:
        model = BARGAINING_MEMORY[key]
        for record in game["game_state"].get("history", []):
            if not isinstance(record, dict):
                continue
            offer = record.get("offer") or {}
            proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
            decision = record.get("decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("bargaining", game.get("game_id"), record.get("round"), proposer,
                     str(decision), tuple(sorted((str(k), str(v)) for k, v in offer.items())))
            if event in SEEN:
                continue
            SEEN.add(event)
            if proposer == canonical_player(me) and str(decision).lower() == "reject":
                rejected = allocation(offer, opponent)
                if rejected is not None and money > 0:
                    model["rejected_floor"] = max(model["rejected_floor"], rejected / money)
            if proposer == canonical_player(opponent):
                demand = allocation(offer, opponent)
                if demand is not None and money > 0:
                    model["opponent_demands"].append(clamp(demand / money, 0.0, 1.0))
        return {"rejected_floor": model["rejected_floor"],
                "opponent_demands": list(model["opponent_demands"])}

def estimate_bargaining_floor(game, state, me, opponent, model):
    t = round_progress(state)
    if state.get("complete_information"):
        prior = rubinstein_responder_share(player_delta(state, me),
                                           player_delta(state, opponent))
        prior = clamp(prior, 0.34, 0.50)
    else:
        opponent_type = (game.get("opponent") or {}).get("type")
        prior = 0.46 if opponent_type == "human" else 0.43
        prior += 0.025 * t
    evidence = model["rejected_floor"] + 0.012 if model["rejected_floor"] else 0.0
    if model["opponent_demands"]:
        recent = model["opponent_demands"][-4:]
        # A proposer usually asks for more than their eventual acceptance floor.
        demand_floor = statistics.median(recent) - (0.075 - 0.025 * t)
        evidence = max(evidence, demand_floor)
    return clamp(max(prior, evidence), 0.30, 0.62)

def bargaining_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    money = finite_float(state["money_to_divide"])
    t = round_progress(state)
    my_delta = player_delta(state, me)
    model = update_bargaining_memory(game, me, opponent, money)
    floor = estimate_bargaining_floor(game, state, me, opponent, model)

    if game["valid_actions"]["type"] == "offer":
        best = None
        failure_cost = 0.035 + 0.22 * t + 0.45 * (1.0 - my_delta)
        for percent in range(28, 66):
            responder_share = percent / 100.0
            probability = logistic((responder_share - floor + 0.012) / 0.022)
            own_share = 1.0 - responder_share
            objective = probability * own_share**1.18 - (1.0 - probability) * failure_cost
            candidate = (objective, own_share)
            if best is None or candidate > best:
                best = candidate
        own_share = clamp(best[1], 0.35, 0.72)
        own_gain = round(money * own_share, 8)
        other_gain = money - own_gain
        action = ({"alice_gain": own_gain, "bob_gain": other_gain}
                  if player_index(me) == 1 else
                  {"alice_gain": other_gain, "bob_gain": own_gain})
        if state.get("messages_allowed"):
            pct = round(100 * other_gain / money) if money else 50
            action["message"] = action_message(
                f"You receive {pct}% now; another round erodes value. I can settle immediately."
            )
        return action

    current_gain = allocation(state.get("last_offer") or {}, me)
    if current_gain is None:
        return {"decision": "reject"}
    if final_round(state):
        return {"decision": "accept" if current_gain >= 0 else "reject"}
    next_own_share = 1.0 - floor
    deal_probability = clamp(0.78 + 0.12 * t - 0.25 * model["rejected_floor"], 0.52, 0.92)
    continuation_share = my_delta * next_own_share * deal_probability
    risk_floor = 0.36 - 0.08 * t
    required = money * max(risk_floor, continuation_share)
    return {"decision": "accept" if current_gain + 1e-9 >= required else "reject"}



## 2. Negotiation — surplus capture with concession inference

Under complete information, every price is converted to a role-specific share of
feasible surplus. Under hidden values, the opponent's own offer becomes a revealed
feasible anchor: V2 multiplied that price beyond the observed bargaining range,
whereas V3 bargains *inside* the interval between its valuation and the opponent's
latest price. Final-round decisions retain individual rationality.



In [ ]:
def offer_sender(item, record, field):
    sender = item.get("from_player") if isinstance(item, dict) else None
    if sender:
        return canonical_player(sender)
    if field == "counteroffer" and record.get("decided_by"):
        return canonical_player(record["decided_by"])
    return None

def update_negotiation_memory(game, state):
    key = opponent_key(game, "negotiation")
    with LOCK:
        model = NEGOTIATION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            for field in ("offer", "counteroffer"):
                item = record.get(field)
                if not isinstance(item, dict) or item.get("price") is None:
                    continue
                sender = offer_sender(item, record, field)
                if sender is None:
                    continue
                price = finite_float(item["price"])
                event = ("negotiation", game.get("game_id"), record.get("round"),
                         field, sender, price)
                if event in SEEN:
                    continue
                SEEN.add(event)
                role = state.get(f"{sender}_role")
                if role in {"seller", "buyer"}:
                    model[f"{role}_prices"].append(price)
        return {name: list(values) for name, values in model.items()}

def opponent_prices_in_game(state, opponent):
    prices = []
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        for field in ("offer", "counteroffer"):
            item = record.get(field)
            if isinstance(item, dict) and item.get("price") is not None:
                if offer_sender(item, record, field) == canonical_player(opponent):
                    prices.append(finite_float(item["price"]))
    last = state.get("last_offer") or {}
    if last.get("price") is not None and canonical_player(last.get("from_player", opponent)) == canonical_player(opponent):
        prices.append(finite_float(last["price"]))
    return prices

def negotiation_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    role = state[f"{me}_role"]
    my_value = finite_float(state[f"{me}_value"])
    t = round_progress(state)
    update_negotiation_memory(game, state)
    observed = opponent_prices_in_game(state, opponent)
    opponent_value = state.get(f"{opponent}_value")
    surplus = None
    seller_value = buyer_value = None

    if state.get("complete_information") and opponent_value is not None:
        opponent_value = finite_float(opponent_value)
        seller_value = my_value if role == "seller" else opponent_value
        buyer_value = my_value if role == "buyer" else opponent_value
        surplus = buyer_value - seller_value
        own_capture = 0.74 - 0.14 * t
        if observed and surplus > 1e-12:
            last_price = observed[-1]
            opponent_role = state[f"{opponent}_role"]
            opponent_demand = ((last_price - seller_value) / surplus if opponent_role == "seller"
                               else (buyer_value - last_price) / surplus)
            feasible_capture = 1.0 - clamp(opponent_demand - (0.05 + 0.04 * t), 0.0, 1.0)
            own_capture = 0.55 * own_capture + 0.45 * feasible_capture
        own_capture = clamp(own_capture, 0.52, 0.82)
        if surplus <= 0:
            target = my_value
        elif role == "seller":
            target = seller_value + own_capture * surplus
        else:
            target = buyer_value - own_capture * surplus
    elif observed:
        anchor = observed[-1]
        claim = 0.70 - 0.12 * t
        if role == "seller" and anchor >= my_value:
            target = my_value + claim * (anchor - my_value)
        elif role == "buyer" and anchor <= my_value:
            target = my_value - claim * (my_value - anchor)
        else:
            target = my_value
    elif role == "seller":
        target = my_value * (1.38 - 0.18 * t)
    else:
        target = my_value * (0.72 + 0.16 * t)

    target = max(0.0, finite_float(target, my_value))
    if game["valid_actions"]["type"] == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = action_message(
                "This price leaves measurable surplus on both sides and avoids losing the deal."
            )
        return action

    price = finite_float((state.get("last_offer") or {}).get("price"), my_value)
    offered_utility = price - my_value if role == "seller" else my_value - price
    profitable = offered_utility >= -1e-9
    if final_round(state):
        return {"decision": "AcceptOffer" if profitable else "RejectOffer"}
    if surplus is not None and surplus > 1e-12:
        offered_capture = offered_utility / surplus
        target_utility = abs(target - my_value)
    else:
        offered_capture = None
        target_utility = abs(target - my_value)
    continuation = target_utility * (0.82 - 0.12 * t)
    if profitable and (offered_utility + 1e-9 >= continuation or
                       (offered_capture is not None and offered_capture >= 0.50 + 0.05 * (1 - t))):
        return {"decision": "AcceptOffer"}

    blend = 0.22 + 0.43 * t
    counter = (1.0 - blend) * target + blend * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    action = {"decision": "RejectOffer", "product_price": round(max(0.0, counter), 8)}
    if state.get("messages_allowed"):
        action["message"] = action_message(
            "I am moving toward you; this counter remains individually rational for me."
        )
    return action



## 3. Persuasion — direct precision learning and credibility budgeting

Buyer observations are censored: quality is normally revealed only after a purchase.
Estimating `P(message | quality)` from those data is biased. V3 instead estimates the
directly useful `P(high | positive message)` (and its negative-message analogue),
shrunk toward a strategic prior. The seller recommends all high products and pools low
products only when the Bayesian threshold, observed buyer response, and accumulated
credibility permit it.



In [ ]:
def update_persuasion_memory(game, seller_view):
    state = game["game_state"]
    key = opponent_key(game, "persuasion")
    with LOCK:
        model = PERSUASION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            signal = signal_polarity(record.get("seller_message"))
            quality = record.get("quality")
            decision = record.get("buyer_decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("persuasion", game.get("game_id"), record.get("round"),
                     signal, quality, str(decision))
            if event in SEEN:
                continue
            SEEN.add(event)
            # The seller knows all qualities; the buyer sees quality only after buying.
            if signal is not None and quality in {"high", "low"}:
                model[f"{'pos' if signal else 'neg'}_{quality}"] += 1.0
            if seller_view and signal is True and str(decision).lower() in {"yes", "no"}:
                model["positive_decisions"] += 1.0
                model["positive_buys"] += float(str(decision).lower() == "yes")
        return dict(model)

def smoothed_signal_precision(model, positive, p):
    if positive:
        high, low = model["pos_high"], model["pos_low"]
        q_high, q_low = 0.90, 0.24
    else:
        high, low = model["neg_high"], model["neg_low"]
        q_high, q_low = 0.10, 0.76
    denominator = p * q_high + (1.0 - p) * q_low
    strategic_prior = p * q_high / denominator if denominator > 1e-12 else p
    strength = 4.0
    return (strength * strategic_prior + high) / (strength + high + low)

def persuasion_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = finite_float(state["product_price"])
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    round_no = max(1, int(state.get("round", 1)))
    total_rounds = max(round_no, int(state.get("total_rounds", round_no)))
    t = (round_no - 1) / max(1, total_rounds - 1)
    seller_view = action_type in {"seller_message", "seller_recommendation"}
    model = update_persuasion_memory(game, seller_view)

    if seller_view:
        quality = state.get("current_quality")
        recommend = quality == "high"
        if quality == "low" and "v" in state and "u" in state:
            v, u = finite_float(state["v"]), finite_float(state["u"])
            if price <= u:
                recommend = True
            elif price >= v or not (0.0 < p < 1.0) or v <= u:
                recommend = False
            else:
                cutoff = clamp((price - u) / (v - u), 1e-6, 1.0 - 1e-6)
                static_pool = p * (1.0 - cutoff) / (cutoff * (1.0 - p))
                response_rate = ((model["positive_buys"] + 1.5) /
                                 (model["positive_decisions"] + 2.0))
                ramp = 0.12 + 0.88 * t**1.65
                pool_probability = clamp(static_pool * ramp * (0.55 + 0.60 * response_rate),
                                         0.0, 1.0)
                prior_high = 2.5
                prior_low = 0.8
                precision_after_lie = ((prior_high + model["pos_high"]) /
                                       (prior_high + prior_low + model["pos_high"] +
                                        model["pos_low"] + 1.0))
                credibility_ok = precision_after_lie >= min(0.97, cutoff + 0.025)
                truthful_prefix_done = round_no > max(1, math.ceil(0.12 * total_rounds))
                recommend = (truthful_prefix_done and credibility_ok and
                             stable_unit(game, "credibility-pool") < pool_probability)
        if action_type == "seller_recommendation":
            return {"decision": "yes" if recommend else "no"}
        return {"message": action_message(
            "My recommendation: BUY this product." if recommend else
            "My recommendation: PASS on this product this round."
        )}

    v, u = finite_float(state["v"]), finite_float(state["u"])
    if price <= u:
        return {"decision": "yes"}
    if price > v:
        return {"decision": "no"}
    signal = signal_polarity(state.get("seller_message"))
    posterior = p if signal is None else smoothed_signal_precision(model, signal, p)
    expected_value = posterior * v + (1.0 - posterior) * u
    observations = (model["pos_high"] + model["pos_low"] if signal is True else
                    model["neg_high"] + model["neg_low"] if signal is False else 0.0)
    remaining_fraction = (total_rounds - round_no) / max(1, total_rounds)
    information_bonus = (0.012 * max(0.0, v - u) * remaining_fraction /
                         math.sqrt(1.0 + observations) if signal is True else 0.0)
    return {"decision": "yes" if expected_value + information_bonus >= price else "no"}



## Validated dispatcher and safe fallback

Invalid moves and turn timeouts are scored at the fifth percentile. Every V3 move is
therefore validated locally; unexpected schemas fall back to a conservative legal
action and are recorded in `DECISION_LOG`.



In [ ]:
STRATEGIES = {
    "bargaining": bargaining_strategy,
    "negotiation": negotiation_strategy,
    "persuasion": persuasion_strategy,
}

def fallback_action(game):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        if action_type == "offer":
            money = finite_float(state["money_to_divide"])
            alice = round(money / 2.0, 8)
            return {"alice_gain": alice, "bob_gain": money - alice}
        return {"decision": "accept"}
    if family == "negotiation":
        me = canonical_player(game.get("your_player", state["current_player"]))
        role = state[f"{me}_role"]
        value = finite_float(state[f"{me}_value"])
        if action_type == "offer":
            return {"product_price": max(0.0, value)}
        price = finite_float((state.get("last_offer") or {}).get("price"), value)
        profitable = price >= value if role == "seller" else price <= value
        if profitable:
            return {"decision": "AcceptOffer"}
        if final_round(state):
            return {"decision": "RejectOffer"}
        return {"decision": "RejectOffer", "product_price": max(0.0, value)}
    if action_type == "seller_message":
        return {"message": "My recommendation: PASS this round."}
    if action_type == "seller_recommendation":
        return {"decision": "no"}
    p = finite_float(state.get("p"), 0.5)
    expected = p * finite_float(state.get("v")) + (1 - p) * finite_float(state.get("u"))
    return {"decision": "yes" if expected >= finite_float(state["product_price"]) else "no"}

def validate_action(game, action):
    if not isinstance(action, dict):
        raise ValueError("strategy must return a dict")
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining" and action_type == "offer":
        alice = finite_float(action["alice_gain"], math.nan)
        bob = finite_float(action["bob_gain"], math.nan)
        pot = finite_float(state["money_to_divide"])
        if not all(math.isfinite(x) and x >= 0 for x in (alice, bob)):
            raise ValueError("invalid bargaining allocation")
        if not math.isclose(alice + bob, pot, rel_tol=1e-10, abs_tol=1e-7):
            raise ValueError("bargaining gains do not sum to the pot")
    elif family == "bargaining":
        if action.get("decision") not in {"accept", "reject", "walkaway"}:
            raise ValueError("invalid bargaining decision")
    elif family == "negotiation" and action_type == "offer":
        if finite_float(action.get("product_price"), -1) < 0:
            raise ValueError("invalid negotiation price")
    elif family == "negotiation":
        if action.get("decision") not in {"AcceptOffer", "RejectOffer", "WalkAway"}:
            raise ValueError("invalid negotiation decision")
        if action["decision"] == "RejectOffer" and not final_round(state):
            if finite_float(action.get("product_price"), -1) < 0:
                raise ValueError("counteroffer required")
    elif action_type == "seller_message":
        if not isinstance(action.get("message"), str) or len(action["message"]) > 2000:
            raise ValueError("invalid persuasion message")
    elif action.get("decision") not in {"yes", "no"}:
        raise ValueError("invalid persuasion decision")
    if "message" in action and len(str(action["message"])) > 2000:
        raise ValueError("message exceeds 2,000 characters")
    return action

def strategy(game):
    error = None
    try:
        family = game["game_family"]
        action = validate_action(game, STRATEGIES[family](game))
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        action = validate_action(game, fallback_action(game))
        print(f"SAFE FALLBACK {game.get('game_id')}: {error}")
    with LOCK:
        DECISION_LOG.append({
            "game_id": game.get("game_id"), "family": game.get("game_family"),
            "round": (game.get("game_state") or {}).get("round"),
            "action": dict(action), "error": error,
        })
    return action



## Offline schema and policy tests

These tests make no API calls. They cover both player positions, exact bargaining
sums, complete and hidden negotiation, final-round legality, binary persuasion, and
censored buyer learning.



In [ ]:
def base_game(family, action_type, state, player="player_1", game_id="test"):
    return {
        "game_id": game_id, "game_family": family, "your_player": player,
        "opponent": {"type": "hidden", "name": None},
        "valid_actions": {"type": action_type, "fields": {}},
        "game_state": state,
    }

def run_smoke_tests():
    for player in ("player_1", "player_2"):
        b_state = {
            "current_player": player, "round": 1, "max_rounds": 5,
            "horizon_known": True, "money_to_divide": 100, "delta_1": 0.9,
            "delta_2": 0.95, "complete_information": True, "history": [],
            "messages_allowed": True,
        }
        action = strategy(base_game("bargaining", "offer", b_state, player,
                                    f"test-b-{player}"))
        assert math.isclose(action["alice_gain"] + action["bob_gain"], 100)
        assert action["alice_gain"] >= 0 and action["bob_gain"] >= 0

    b_decision_state = {
        "current_player": "player_2", "round": 5, "max_rounds": 5,
        "horizon_known": True, "money_to_divide": 100, "complete_information": False,
        "last_offer": {"player_1_gain": 99, "player_2_gain": 1}, "history": [],
    }
    assert strategy(base_game("bargaining", "decision", b_decision_state,
                              "player_2", "test-b-final"))["decision"] == "accept"

    n_state = {
        "current_player": "player_1", "player_1_role": "seller",
        "player_2_role": "buyer", "player_1_value": 40, "player_2_value": 100,
        "complete_information": True, "round": 1, "max_rounds": 5,
        "horizon_known": True, "history": [], "messages_allowed": True,
    }
    price = strategy(base_game("negotiation", "offer", n_state,
                               "player_1", "test-n-full"))["product_price"]
    assert 40 <= price <= 100

    n_hidden = dict(n_state, current_player="player_2", player_2_value=100,
                    complete_information=False, round=2,
                    last_offer={"price": 70, "from_player": "player_1"},
                    history=[{"round": 1, "offer": {"price": 70,
                             "from_player": "player_1"}}])
    n_hidden.pop("player_1_value")
    counter = strategy(base_game("negotiation", "decision", n_hidden,
                                 "player_2", "test-n-hidden"))
    assert counter["decision"] in {"AcceptOffer", "RejectOffer"}
    if counter["decision"] == "RejectOffer":
        assert 0 <= counter["product_price"] <= 100

    p_seller = {
        "current_quality": "high", "product_price": 50, "p": 0.5,
        "v": 100, "u": 0, "round": 1, "total_rounds": 10, "history": [],
    }
    assert strategy(base_game("persuasion", "seller_recommendation", p_seller,
                              "player_1", "test-p-seller")) == {"decision": "yes"}

    p_buyer = {
        "seller_message": {"decision": "yes"}, "product_price": 45, "p": 0.5,
        "v": 100, "u": 0, "round": 3, "total_rounds": 10,
        "history": [{"round": 1, "seller_message": {"decision": "yes"},
                     "buyer_decision": "yes", "quality": "high"}],
    }
    assert strategy(base_game("persuasion", "buyer_decision", p_buyer,
                              "player_2", "test-p-buyer"))["decision"] in {"yes", "no"}
    assert not [entry for entry in DECISION_LOG if entry["error"]]
    print("All V3 smoke tests passed.")

run_smoke_tests()



## Play a controlled evaluation batch

Start with small, separate family batches. Compare rating *and* games played before
and after each batch; displayed rating is shrunk toward 1,000 by `g / (g + 30)`, so
compare enough games and do not attribute every short-run fluctuation to policy.
Increase concurrency only after the log shows no fallbacks. The SDK drains active
games before returning, preventing timeout penalties.



In [ ]:
from glee_sdk import GleeClient

GAME_FAMILIES = ["bargaining", "negotiation", "persuasion"]
CONCURRENCY = 3
MAX_GAMES = 30
MAX_TIME = 3600

client = GleeClient(api_key=os.environ["GLEE_API_KEY"])
before = client.stats()
print("Before:", before)
client.run(
    strategy,
    game_families=GAME_FAMILIES,
    concurrency=CONCURRENCY,
    max_games=MAX_GAMES,
    max_time=MAX_TIME,
)
after = client.stats()
print("After:", after)



## Inspect decisions and fallbacks



In [ ]:
fallbacks = [entry for entry in DECISION_LOG if entry["error"]]
print("Fallback count:", len(fallbacks))
print("Recent decisions:")
list(DECISION_LOG)[-20:]

